<a href="https://colab.research.google.com/github/imid12/neuraleagleshcc/blob/main/final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

    1. Loaded the dataset: We started by reading the sleep_cycle_productivity.csv file.
    2. Preprocessed the data (as per your initial instruction): While you mentioned this was already done, we implicitly handled potential issues when dropping the 'Date' and 'Gender' columns.
    3. Split the data: We divided the dataset into training (70%), validation (15%), and test (15%) sets. The validation set was crucial for evaluating and comparing different models and for building our ensemble. The test set was reserved for the final evaluation of our best performing model(s).
    4. Trained individual regression models: We trained five different regression models: Linear Regression, Decision Tree Regressor, Random Forest Regressor, Gradient Boosting Regressor, and K-Nearest Neighbors Regressor.
    5. Validated and compared the models: We used the validation set to calculate the Mean Absolute Error (MAE), Mean Squared Error (MSE), and R² Score for each of the trained models. This allowed us to understand how well each model generalized to unseen data and to identify the top performers.
    6. Built ensemble models:
       * Voting Regressor: We created a Voting Regressor by combining the predictions of the top 3 models based on their performance on the validation set.
       * Averaged Predictions: We also averaged the predictions of the same top 3 models.
       * Bayesian-inspired Averaging: We implemented a weighted averaging approach, where the weights were determined by the inverse of the MSE of each model on the validation set, giving more influence to better-performing models.
    7. Evaluated ensemble models: We assessed the performance of the Voting Regressor and the averaged predictions on both the validation and the test sets to get an estimate of their real-world performance.

So, yes, the entire process was geared towards accurately predicting the "Productivity Score" using various regression techniques and ensemble methods. The final evaluation metrics on the test set for the ensemble models (Voting and Bayesian-inspired Averaging) would give you an indication of how well we can predict this score on new, unseen data.

Do you have any specific questions about the prediction of the "Productivity Score" or any particular aspect of the modeling process? Perhaps you'd like to look at the feature importances from a model like Random Forest or Gradient Boosting to understand which factors have the most influence on the predicted score? Let me know what you'd like to explore further!

In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, VotingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# Load the dataset
df = pd.read_csv('sleep_cycle_productivity.csv')

# Show the different features
df.describe()

# Drop the 'Date' feature
if 'Date' in df.columns:
    df = df.drop('Date', axis=1)
    print("Dropped the 'Date' feature.")
else:
    print("'Date' feature not found in the dataset.")

# Drop the 'Gender' feature
if 'Gender' in df.columns:
    df = df.drop('Gender', axis=1)
    print("Dropped the 'Gender' feature.")
else:
    print("'Gender' feature not found in the dataset.")

# Assuming 'Productivity Score' is the target variable
X = df.drop('Productivity Score', axis=1)
y = df['Productivity Score']

# Split data into training (70%), validation (15%), and test (15%)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)


# Save the training set
train_df = pd.concat([X_train, y_train], axis=1)
train_df.to_csv('train_data.csv', index=False)
print("Training data saved to 'train_data.csv'")

# Save the validation set
val_df = pd.concat([X_val, y_val], axis=1)
val_df.to_csv('validation_data.csv', index=False)
print("Validation data saved to 'validation_data.csv'")

# Save the test set
test_df = pd.concat([X_test, y_test], axis=1)
test_df.to_csv('test_data.csv', index=False)
print("Test data saved to 'test_data.csv'")

print(f"Training set size: {len(X_train)}")
print(f"Validation set size: {len(X_val)}")
print(f"Test set size: {len(X_test)}")

# Initialize the models
linear_reg = LinearRegression()
decision_tree = DecisionTreeRegressor(random_state=42)
random_forest = RandomForestRegressor(random_state=42)
gradient_boosting = GradientBoostingRegressor(random_state=42)
knn = KNeighborsRegressor()

# Train the models
linear_reg.fit(X_train, y_train)
decision_tree.fit(X_train, y_train)
random_forest.fit(X_train, y_train)
gradient_boosting.fit(X_train, y_train)
knn.fit(X_train, y_train)

# Make predictions on the validation set
y_pred_lr = linear_reg.predict(X_val)
y_pred_dt = decision_tree.predict(X_val)
y_pred_rf = random_forest.predict(X_val)
y_pred_gb = gradient_boosting.predict(X_val)
y_pred_knn = knn.predict(X_val)

# Evaluate the models on the validation set
def evaluate_model(y_true, y_pred, model_name):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"----- {model_name} -----")
    print(f"MAE: {mae:.4f}")
    print(f"MSE: {mse:.4f}")
    print(f"R² Score: {r2:.4f}")
    return mae, mse, r2

lr_metrics = evaluate_model(y_val, y_pred_lr, "Linear Regression")
dt_metrics = evaluate_model(y_val, y_pred_dt, "Decision Tree Regressor")
rf_metrics = evaluate_model(y_val, y_pred_rf, "Random Forest Regressor")
gb_metrics = evaluate_model(y_val, y_pred_gb, "Gradient Boosting Regressor")
knn_metrics = evaluate_model(y_val, y_pred_knn, "K-Nearest Neighbors Regressor")

# Collect the validation metrics
validation_metrics = {
    "Linear Regression": lr_metrics,
    "Decision Tree Regressor": dt_metrics,
    "Random Forest Regressor": rf_metrics,
    "Gradient Boosting Regressor": gb_metrics,
    "K-Nearest Neighbors Regressor": knn_metrics
}

# Sort models by R² score (higher is better)
sorted_models_r2 = sorted(validation_metrics.items(), key=lambda item: item[1][2], reverse=True)
top_3_models_r2 = [name for name, metrics in sorted_models_r2[:3]]
print("\nTop 3 models based on Validation R² Score:", top_3_models_r2)

# Create a Voting Regressor with the top 3 models (based on R² for this example)
estimators = []
if "Random Forest Regressor" in top_3_models_r2:
    estimators.append(('rf', random_forest))
if "Gradient Boosting Regressor" in top_3_models_r2:
    estimators.append(('gb', gradient_boosting))
if "Linear Regression" in top_3_models_r2:
    estimators.append(('lr', linear_reg))
if "Decision Tree Regressor" in top_3_models_r2:
    estimators.append(('dt', decision_tree))
if "K-Nearest Neighbors Regressor" in top_3_models_r2:
    estimators.append(('knn', knn))

voting_reg = VotingRegressor(estimators)
voting_reg.fit(X_train, y_train)

# Make predictions with the Voting Regressor on the validation and test sets
y_pred_vote_val = voting_reg.predict(X_val)
y_pred_vote_test = voting_reg.predict(X_test)

# Evaluate the Voting Regressor
print("\n----- Voting Regressor (Validation Set) -----")
evaluate_model(y_val, y_pred_vote_val, "Voting Regressor")

print("\n----- Voting Regressor (Test Set) -----")
evaluate_model(y_test, y_pred_vote_test, "Voting Regressor")

# Average predictions of the top 3 models
top_3_predictions_val = []
top_3_predictions_test = []

if "Random Forest Regressor" in top_3_models_r2:
    top_3_predictions_val.append(y_pred_rf)
    top_3_predictions_test.append(random_forest.predict(X_test))
if "Gradient Boosting Regressor" in top_3_models_r2:
    top_3_predictions_val.append(y_pred_gb)
    top_3_predictions_test.append(gradient_boosting.predict(X_test))
if "Linear Regression" in top_3_models_r2:
    top_3_predictions_val.append(y_pred_lr)
    top_3_predictions_test.append(linear_reg.predict(X_test))
if "Decision Tree Regressor" in top_3_models_r2:
    top_3_predictions_val.append(y_pred_dt)
    top_3_predictions_test.append(decision_tree.predict(X_test))
if "K-Nearest Neighbors Regressor" in top_3_models_r2:
    top_3_predictions_val.append(y_pred_knn)
    top_3_predictions_test.append(knn.predict(X_test))

y_pred_avg_val = np.mean(top_3_predictions_val, axis=0) if top_3_predictions_val else np.zeros_like(y_val)
y_pred_avg_test = np.mean(top_3_predictions_test, axis=0) if top_3_predictions_test else np.zeros_like(y_test)

# Evaluate the Averaged predictions
print("\n----- Averaged Predictions (Validation Set) -----")
evaluate_model(y_val, y_pred_avg_val, "Averaged Predictions")

print("\n----- Averaged Predictions (Test Set) -----")
evaluate_model(y_test, y_pred_avg_test, "Averaged Predictions")

# Bayesian-inspired Averaging (weighting by inverse of MSE on validation set)
weights = {}
if "Random Forest Regressor" in validation_metrics:
    weights["Random Forest Regressor"] = 1 / validation_metrics["Random Forest Regressor"][1]
if "Gradient Boosting Regressor" in validation_metrics:
    weights["Gradient Boosting Regressor"] = 1 / validation_metrics["Gradient Boosting Regressor"][1]
if "Linear Regression" in validation_metrics:
    weights["Linear Regression"] = 1 / validation_metrics["Linear Regression"][1]
if "Decision Tree Regressor" in validation_metrics:
    weights["Decision Tree Regressor"] = 1 / validation_metrics["Decision Tree Regressor"][1]
if "K-Nearest Neighbors Regressor" in validation_metrics:
    weights["K-Nearest Neighbors Regressor"] = 1 / validation_metrics["K-Nearest Neighbors Regressor"][1]

# Normalize the weights
sum_weights = sum(weights.values())
normalized_weights = {k: v / sum_weights for k, v in weights.items()}
print("\nBayesian-inspired Averaging Weights:", normalized_weights)

# Calculate weighted average on validation set
weighted_sum_val = np.zeros_like(y_val, dtype=float)
for model_name, weight in normalized_weights.items():
    if model_name == "Random Forest Regressor":
        weighted_sum_val += weight * y_pred_rf
    elif model_name == "Gradient Boosting Regressor":
        weighted_sum_val += weight * y_pred_gb
    elif model_name == "Linear Regression":
        weighted_sum_val += weight * y_pred_lr
    elif model_name == "Decision Tree Regressor":
        weighted_sum_val += weight * y_pred_dt
    elif model_name == "K-Nearest Neighbors Regressor":
        weighted_sum_val += weight * y_pred_knn
y_pred_bayesian_val = weighted_sum_val

# Calculate weighted average on test set
weighted_sum_test = np.zeros_like(y_test, dtype=float)
for model_name, weight in normalized_weights.items():
    if model_name == "Random Forest Regressor":
        weighted_sum_test += weight * random_forest.predict(X_test)
    elif model_name == "Gradient Boosting Regressor":
        weighted_sum_test += weight * gradient_boosting.predict(X_test)
    elif model_name == "Linear Regression":
        weighted_sum_test += weight * linear_reg.predict(X_test)
    elif model_name == "Decision Tree Regressor":
        weighted_sum_test += weight * decision_tree.predict(X_test)
    elif model_name == "K-Nearest Neighbors Regressor":
        weighted_sum_test += weight * knn.predict(X_test)
y_pred_bayesian_test = weighted_sum_test

# Evaluate the Bayesian-inspired Averaging
print("\n----- Bayesian-inspired Averaging (Validation Set) -----")
evaluate_model(y_val, y_pred_bayesian_val, "Bayesian-inspired Averaging")

print("\n----- Bayesian-inspired Averaging (Test Set) -----")
evaluate_model(y_test, y_pred_bayesian_test, "Bayesian-inspired Averaging")


Dropped the 'Date' feature.
Dropped the 'Gender' feature.
Training data saved to 'train_data.csv'
Validation data saved to 'validation_data.csv'
Test data saved to 'test_data.csv'
Training set size: 3500
Validation set size: 750
Test set size: 750
----- Linear Regression -----
MAE: 2.5641
MSE: 8.4682
R² Score: -0.0054
----- Decision Tree Regressor -----
MAE: 3.2813
MSE: 16.2760
R² Score: -0.9323
----- Random Forest Regressor -----
MAE: 2.5986
MSE: 8.7735
R² Score: -0.0416
----- Gradient Boosting Regressor -----
MAE: 2.5995
MSE: 8.7512
R² Score: -0.0390
----- K-Nearest Neighbors Regressor -----
MAE: 2.7475
MSE: 10.3873
R² Score: -0.2332

Top 3 models based on Validation R² Score: ['Linear Regression', 'Gradient Boosting Regressor', 'Random Forest Regressor']

----- Voting Regressor (Validation Set) -----
----- Voting Regressor -----
MAE: 2.5819
MSE: 8.5821
R² Score: -0.0189

----- Voting Regressor (Test Set) -----
----- Voting Regressor -----
MAE: 2.5178
MSE: 8.3739
R² Score: -0.0227

-

(2.515136661723006, 8.454016895281525, -0.03252584332320452)

Understanding the Metrics:

###    * **Mean Absolute Error (MAE):** This measures the average magnitude of the errors between your model's predictions and the actual values. It's in the same units as your target variable, making it relatively easy to interpret. A lower MAE indicates better performance.   

###    * **Mean Squared Error (MSE):** This measures the average of the squared differences between your model's predictions and the actual values. Squaring the errors penalizes larger errors more heavily than MAE. The units of MSE are the squared units of your target variable, making direct interpretation slightly less intuitive. Lower MSE indicates better performance.  

###    * **R² Score (Coefficient of Determination)**: This represents the proportion of the variance in the dependent variable that is predictable from the independent variables. It ranges from 0 to 1 (or can be negative if the model performs worse than a simple average).  

      * An R² of 1 indicates that the model explains all the variance in the target variable.   
      * An R² of 0 suggests that the model does not explain any of the variance.  
      * A higher R² generally indicates a better fit of the model to the data.

In [16]:
# Compare Bayesian Ensemble with Linear Regression
linear_regression_predictions = linear_reg.predict(X_test) # Access the Linear Regression model object (linear_reg) directly
mae_linear_test = mean_absolute_error(y_test, linear_regression_predictions)
mse_linear_test = mean_squared_error(y_test, linear_regression_predictions)
r2_linear_test = r2_score(y_test, linear_regression_predictions)

# Calculate MAE, MSE, and R2 for the Bayesian Ensemble on the test set
mae_bayesian_test = mean_absolute_error(y_test, y_pred_bayesian_test)  # Calculate MAE for Bayesian Ensemble
mse_bayesian_test = mean_squared_error(y_test, y_pred_bayesian_test)  # Calculate MSE for Bayesian Ensemble
r2_bayesian_test = r2_score(y_test, y_pred_bayesian_test)  # Calculate R2 for Bayesian Ensemble


print("\nLinear Regression Model Results (Test Set):")
print(f"MAE = {mae_linear_test:.2f}, MSE = {mse_linear_test:.2f}, R2 = {r2_linear_test:.2f}")

print("\nComparison of Bayesian Ensemble and Linear Regression:")
print(f"Bayesian Ensemble MAE: {mae_bayesian_test:.2f}, Linear Regression MAE: {mae_linear_test:.2f}")
print(f"Bayesian Ensemble MSE: {mse_bayesian_test:.2f}, Linear Regression MSE: {mse_linear_test:.2f}")
print(f"Bayesian Ensemble R2: {r2_bayesian_test:.2f}, Linear Regression R2: {r2_linear_test:.2f}")


Linear Regression Model Results (Test Set):
MAE = 2.51, MSE = 8.28, R2 = -0.01

Comparison of Bayesian Ensemble and Linear Regression:
Bayesian Ensemble MAE: 2.52, Linear Regression MAE: 2.51
Bayesian Ensemble MSE: 8.45, Linear Regression MSE: 8.28
Bayesian Ensemble R2: -0.03, Linear Regression R2: -0.01


**Picked the Linear Regression Model because it was in the top 3 models and it had a lower MSE and MAE so it would perform better than the other models.**